 # Pipeline

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import librosa
import torch


/home/garifzjanovni/.conda/envs/ml_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda:3'

In [3]:
processor = WhisperProcessor.from_pretrained("openai/whisper-large")

In [4]:

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large")
model = model.to(device)

In [ ]:
test_audio, sr = librosa.load('meeting.wav', sr=16_000)
inputs = processor(
    test_audio,
    return_tensors="pt",
    truncation=False,
    padding="longest",
    return_attention_mask=True,
    sampling_rate=16_000
)

In [ ]:
generated_ids = model.generate(return_timestamps=True, language="ru", **inputs.to(device))

You have passed language=ru, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of language=ru.


In [ ]:
decoded = processor.batch_decode(generated_ids.cpu())
transcription = decoded[0]


' в рамках продуктов сутки будущего по искусственному интеллекту, посмотреть на них с точки зрения того, что получается, моменты коллеги подсветят, которые вызывают сейчас вопросы, понять, как мы можем совместно их решить, может быть появится новое управление и здесь своей стороны сформировать по итогу какой-то план реализации запросов, либо выделить те направления, которые мешают искусственному интеллекту поскорее внедряться. У нас по порядку по продуктам пойдет. Смотрите, я бы хотел, чтобы отношение было окончательно с тем, что мы говорим о цедравящем, интеллекту, это не... пока это воспринимается как хайп, особенно если нас оберечают глупые наши компании, которые считают, что это там просто мода, как очередная, не знаю, фишка какая-то пройдет и все, но я вот не верю во что. Мне очень зашло выступление одного этого лицепрезидента Эльфеиса Схамажева, который я его сбросил на орунник, как он мне считал, через Энкар, и в общем я я могу сбросить, я бы разбрасывал, но по второму пробу не 

In [ ]:
with open('transcription.txt', 'w+') as f:
    f.write(transcription[0])

---

In [ ]:
from tqdm import tqdm
with open('transcription.txt', 'r') as f:
    transcription = f.read()

del model

In [ ]:
import torch
from transformers import GPT2Tokenizer, T5ForConditionalGeneration 
tokenizer = GPT2Tokenizer.from_pretrained('RussianNLP/FRED-T5-Summarizer',eos_token='</s>')
model = T5ForConditionalGeneration.from_pretrained('RussianNLP/FRED-T5-Summarizer')


In [ ]:
device = torch.device('cuda')
model = model.to(device)

In [ ]:
def chunk_text(text, max_tokens=1024):
    sentences = text.split('. ')
    chunks, chunk = [], ''
    current_len = 0

    for sentence in sentences:
        token_len = len(tokenizer.encode(sentence, add_special_tokens=False))
        if current_len + token_len > max_tokens:
            chunks.append(chunk.strip())
            chunk = sentence + '. '
            current_len = token_len
        else:
            chunk += sentence + '. '
            current_len += token_len
    if chunk:
        chunks.append(chunk.strip())
    return chunks

def tokenize_chunk(chunk):
    input_text = "<LM> Сократи текст.\n " + chunk
    input_ids = torch.tensor([tokenizer.encode(input_text)]).to(device)
    return input_ids

def summarize_chunk(input_ids):
    summary_ids = model.generate(
        input_ids,
        eos_token_id=tokenizer.eos_token_id,
        num_beams=5,
        min_new_tokens=17,
        max_new_tokens=200,
        do_sample=True,
        no_repeat_ngram_size=4,
        top_p=0.9
    )
    return tokenizer.decode(summary_ids[0][1:])

In [ ]:
chunks = chunk_text(transcription)
tokenized_chunks = [tokenize_chunk(c) for c in chunks]

In [ ]:
summaries = [summarize_chunk(c) for c in tokenized_chunks]

In [ ]:
with open('summary.txt', 'w+') as f:
    f.write('\n'.join(summaries))